# Engenharia de atributos e boosting em série temporal

Os dois notebooks anteriores usaram os métodos que já sabem que a série tem ordem: ETS trata nível,
tendência e sazonalidade diretamente, e SARIMAX modela a dependência temporal nos próprios parâmetros.

Este notebook faz o caminho oposto, e é o mais usado na prática: **transformar a série numa tabela comum**
e entregá-la a um modelo que não sabe nada de tempo. Cada linha é um dia, cada coluna é uma informação
sobre os dias anteriores ou sobre o calendário daquele dia, e a partir daí vale tudo que já se sabe de
aprendizado supervisionado.

Quatro decisões vêm junto, e todas se medem:

1. **quais atrasos (lags)** entram, e quantos;
2. **quais atributos de calendário** entram, e o risco de usar o dia do ano;
3. **a regra do `shift`**: janela móvel sem deslocamento coloca a resposta dentro do preditor;
4. **recursivo ou direto**: um modelo que consome as próprias previsões, ou um modelo por passo.

No fim, o confronto direto com os clássicos e com o sarrafo do sazonal ingênuo, na mesma janela de
avaliação. A série é a mesma dos dois notebooks anteriores.


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold, cross_val_score
from statsmodels.tsa.api import ExponentialSmoothing, SARIMAX

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (11, 4)})
warnings.filterwarnings("ignore")  # o SARIMAX avisa muito e o notebook anterior já tratou disso

RANDOM_STATE = 42
PERIODO = 7
ATRASOS = (1, 2, 7, 14, 21, 28)


def serie_com_componentes(n_dias=760, seed=RANDOM_STATE):
    """A mesma série dos notebooks anteriores: tendência, semana, ano, ruído e um choque."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_dias)
    indice = pd.date_range("2024-01-01", periods=n_dias, freq="D")
    valores = (100 + 0.06 * t
               + 6.0 * np.sin(2 * np.pi * t / 7)
               + 12.0 * np.sin(2 * np.pi * t / 365)
               + rng.normal(0, 3.0, n_dias)
               + np.where((t >= 400) & (t < 415), 25.0 * np.exp(-(t - 400) / 4.0), 0.0))
    return pd.Series(valores, index=indice, name="vendas")


def erro_absoluto_medio(observado, previsto):
    return float(np.mean(np.abs(np.asarray(observado) - np.asarray(previsto))))


def mase(observado, previsto, treino, periodo=PERIODO):
    return erro_absoluto_medio(observado, previsto) / float(np.mean(np.abs(treino.diff(periodo).dropna())))


def backtest(serie, previsor, origens=9, passos=14):
    por_origem, erro_por_passo = [], []
    for origem in range(len(serie) - passos * origens, len(serie) - passos, passos):
        treino, alvo = serie.iloc[:origem], serie.iloc[origem:origem + passos]
        previsto = previsor(treino, passos)
        por_origem.append({"MAE": erro_absoluto_medio(alvo, previsto),
                           "MASE": mase(alvo, previsto, treino)})
        erro_por_passo.append(np.abs(alvo.to_numpy() - np.asarray(previsto)))
    return pd.DataFrame(por_origem), np.vstack(erro_por_passo)


serie = serie_com_componentes()
treino, teste = serie.iloc[:-14], serie.iloc[-14:]
print(f"série: {len(serie)} dias | treino até {treino.index[-1].date()} | teste: {len(teste)} dias")
print(f"sarrafo: sazonal ingênuo com MAE 4,332 no recorte e 4,040 no backtesting")
print(f"clássicos: ETS com MAE 2,956 no recorte e 2,947 no backtesting")


## De série para tabela

A conversão é sempre a mesma ideia: a linha do dia `t` recebe o valor dos dias anteriores (`lag_1`, `lag_2`,
`lag_7`, ...) e as informações do próprio dia que se pode saber de antemão, como o dia da semana e o mês.

Duas regras fazem a diferença entre isso funcionar e isso mentir:

- **todo atributo precisa existir no momento da previsão.** Se para prever o dia `t` você precisa de um
  dado que só aparece no dia `t`, o atributo é inútil no mundo real, por mais que ajude no treino;
- **a linha do dia `t` não pode conter `t`.** Vale para lags (por isso `shift`) e para janelas móveis (por
  isso o `shift` antes do `rolling`).

A função abaixo constrói a tabela com os atrasos, o calendário e as janelas móveis, sempre deslocadas.


In [ ]:
def linha_de_atributos(historico, quando, com_dia_do_ano=False, com_janelas=True, vazada=False):
    """As colunas de UM dia, montadas a partir do histórico disponível naquele momento."""
    linha = {f"lag_{atraso}": float(historico.iloc[-atraso]) for atraso in ATRASOS}
    linha["dia_da_semana"] = quando.dayofweek
    linha["mes"] = quando.month
    if com_dia_do_ano:
        linha["dia_do_ano"] = quando.dayofyear
    if com_janelas:
        # a janela termina ontem: `shift(1)` antes de agregar, nunca depois
        passada = historico if vazada else historico.shift(1).dropna()
        linha["media_7"] = float(passada.iloc[-7:].mean())
        linha["desvio_7"] = float(passada.iloc[-7:].std())
    return linha


def tabela_de_atributos(serie, com_dia_do_ano=False, com_janelas=True, vazada=False):
    """Para cada dia, o histórico termina ONTEM: `iloc[:i - 1]` nunca inclui o dia da linha."""
    linhas = [linha_de_atributos(serie.iloc[:i - 1], serie.index[i - 1],
                                 com_dia_do_ano, com_janelas, vazada)
              for i in range(max(ATRASOS) + 1, len(serie) + 1)]
    tabela = pd.DataFrame(linhas, index=serie.index[max(ATRASOS):])
    tabela["valor"] = serie.loc[tabela.index].to_numpy()
    return tabela


tabela = tabela_de_atributos(treino)
print(f"tabela de atributos do treino: {tabela.shape[0]} linhas x {tabela.shape[1] - 1} colunas")
print(f"colunas: {', '.join(c for c in tabela.columns if c != 'valor')}")
display(tabela.head(3).round(2))


## Os atributos de calendário, e o dia do ano

`dia_da_semana` e `mes` são seguros: o valor de hoje é conhecido hoje. `dia_do_ano` é sedutor e arriscado,
porque ele quase identifica a data: com dois anos de dados, o dia 100 de um ano e o dia 100 do outro têm
valores de níveis diferentes, e o modelo pode aprender a memorizar o ano em vez de aprender a sazonalidade.
Em série longa, com muitos anos, ele funciona bem.

A regra deste notebook é não declarar: medir. Os três conjuntos de atributos vão para o mesmo backtesting.


In [ ]:
configuracoes = {
    "atrasos + calendário": dict(com_dia_do_ano=False),
    "atrasos + calendário + dia do ano": dict(com_dia_do_ano=True),
    "atrasos + calendário, sem janelas móveis": dict(com_dia_do_ano=False, com_janelas=False),
}


def previsor_recursivo(treino, horizonte, **config):
    """Um único modelo, alimentado com as próprias previsões a cada passo."""
    tabela_treino = tabela_de_atributos(treino, **config)
    alvo = tabela_treino.pop("valor")
    modelo = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    modelo.fit(tabela_treino, alvo)

    historico, previsoes = treino.copy(), []
    for _ in range(horizonte):
        proximo = historico.index[-1] + pd.Timedelta(days=1)
        linha = pd.DataFrame([linha_de_atributos(historico, proximo, **config)])
        previsao = float(modelo.predict(linha)[0])
        previsoes.append(previsao)
        historico.loc[proximo] = previsao
    return np.array(previsoes)


linhas = []
passos_por_config = {}
for nome, config in configuracoes.items():
    t0 = time.perf_counter()
    resultado, passos = backtest(serie, lambda t, h, c=config: previsor_recursivo(t, h, **c))
    passos_por_config[nome] = passos
    linhas.append({"conjunto de atributos": nome, "MAE médio": passos.mean(),
                   "MAE no passo 1": passos[:, 0].mean(), "MAE no passo 14": passos[:, -1].mean(),
                   "segundos": time.perf_counter() - t0})

tabela_atributos = pd.DataFrame(linhas).set_index("conjunto de atributos").round(3)
display(tabela_atributos)

melhor_config = tabela_atributos["MAE médio"].idxmin()
passos_boosting = passos_por_config[melhor_config]
print(f"melhor conjunto pelo MAE médio: {melhor_config}")


O dia do ano entrou como candidato e **passou no teste**: com ele o MAE médio cai de 3,981 para 3,415, e o
erro no passo 1 cai de 3,857 para 3,080. O receio era que o modelo memorizasse o ano, e nesta série ele não
memorizou, provavelmente porque o gerador declara uma sazonalidade anual de verdade, que o dia do ano
descreve bem. Em série com poucos anos e tendência mais forte, a resposta pode ser a oposta, e é por isso
que a decisão se mede em vez de se supor.

Tirar as janelas móveis custa pouco erro (4,046 contra 3,981) e economiza tempo de ajuste (1,4 s contra
2,3 s numa passada de backtesting). É o tipo de troca que vale saber que existe quando o dado é grande ou a
rodada precisa ser rápida.

## A regra do `shift`, e os dois jeitos de errar

Duas coisas diferentes são chamadas de vazamento em série temporal:

- **atributo que contém a resposta**: uma coluna que só existe depois do fato, como o valor do próprio dia
  que se quer prever. O modelo aprende a ler a resposta, e a nota fica espetacular;
- **janela sem deslocamento**: esquecer o `shift(1)` antes do `rolling`, o que coloca o dia de hoje dentro
  da média que deveria descrever só o passado.

O primeiro é fácil de medir, e a medição mostra o tamanho do estrago. O segundo é mais traiçoeiro: ele pode
quase não mudar o resultado, e é aí que ele é perigoso.


In [ ]:
def previsor_vazado(treino, horizonte):
    return previsor_recursivo(treino, horizonte, vazada=True)


_, passos_honesto = backtest(serie, previsor_recursivo)
_, passos_vazado = backtest(serie, previsor_vazado)


def notas_sorteadas(tabela):
    """O MAE de uma validação cruzada com as linhas sorteadas: o jeito como o vazamento passa."""
    alvo = tabela.pop("valor")
    notas = cross_val_score(HistGradientBoostingRegressor(random_state=RANDOM_STATE), tabela, alvo,
                            cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                            scoring="neg_mean_absolute_error")
    return -notas.mean()


tabela_com_resposta = tabela_de_atributos(treino)
tabela_com_resposta["valor_hoje"] = tabela_com_resposta["valor"]  # a resposta disfarçada de preditor

print("validação cruzada sorteada:")
print(f"  tabela honesta                          MAE {notas_sorteadas(tabela_de_atributos(treino)):7.3f}")
print(f"  com o valor de hoje entre os preditores MAE {notas_sorteadas(tabela_com_resposta):7.3f}")
print()
print("janela móvel deslocada ou não, medido no backtesting:")
print(f"  com shift(1), termina ontem             MAE {passos_honesto.mean():7.3f}")
print(f"  sem shift(1), inclui hoje               MAE {passos_vazado.mean():7.3f}")


A primeira medição é o retrato do vazamento clássico: com o valor do próprio dia entre os preditores, o MAE
sorteado desaba, e não existe previsão nenhuma ali. No dia da previsão essa coluna não existe, e nenhuma
validação que sorteie linhas vai reclamar.

A segunda medição é a parte desconfortável, e vale ser dita: esquecer o `shift` mudou quase nada. Nesta
série o `lag_1` já carrega a informação que a janela vazada acrescenta, então o bug fica invisível.
**É por isso que a regra não se substitui pela métrica**: o erro mais comum de engenharia de atributos em
série é silencioso, e quem só olha o número final não descobre.

## Recursivo ou direto

O previsor recursivo treina **um** modelo e o usa `h` vezes, alimentando-o com as próprias previsões. A
alternativa é treinar **h** modelos, cada um especializado em um passo do horizonte, e cada um usando apenas
os atributos disponíveis na origem. O custo muda, e o erro também.


In [ ]:
def previsor_direto(treino, horizonte):
    """Um modelo por passo do horizonte, todos treinados na mesma origem."""
    tabela_treino = tabela_de_atributos(treino)
    base = tabela_treino.drop(columns="valor")
    previsoes = []
    for passo in range(1, horizonte + 1):
        alvo = tabela_treino["valor"].shift(-passo)
        valido = alvo.notna()
        modelo = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
        modelo.fit(base[valido], alvo[valido])
        quando = treino.index[-1] + pd.Timedelta(days=passo)
        linha = pd.DataFrame([linha_de_atributos(treino, quando)])
        previsoes.append(float(modelo.predict(linha)[0]))
    return np.array(previsoes)


t0 = time.perf_counter()
_, passos_direto = backtest(serie, previsor_direto)
tempo_direto = time.perf_counter() - t0

print("                      MAE médio   passo 1   passo 14")
print(f"  recursivo             {passos_honesto.mean():7.3f}   {passos_honesto[:, 0].mean():7.3f}   "
      f"{passos_honesto[:, -1].mean():7.3f}")
print(f"  direto                {passos_direto.mean():7.3f}   {passos_direto[:, 0].mean():7.3f}   "
      f"{passos_direto[:, -1].mean():7.3f}")
print()
print(f"o direto treina {passos_direto.shape[1]} modelos por origem, o recursivo treina 1: "
      f"{tempo_direto:.1f}s contra ~{tempo_direto / passos_direto.shape[1]:.1f}s de um ajuste só")


Nesta série o recursivo ganhou com folga: MAE médio 3,981 contra 5,645 do direto, e a distância é ainda maior
no primeiro passo (3,857 contra 4,870). A explicação está no dado disponível para treinar: o recursivo
treina **um** modelo com todas as 718 linhas, e o direto treina 14 modelos, cada um com uma resposta
diferente no alvo e todos com as mesmas 718 linhas, o que divide a informação em vez de somar.

O custo caminha junto: o direto gastou 19,0 segundos no backtesting contra cerca de 1,4 segundo de um ajuste
do recursivo, ou seja, catorze vezes mais. A literatura não tem vencedor universal nessa escolha (a revisão
de Taieb e colegas compara as duas em várias competições e encontra resultados que dependem da série), e a
razão prática é simples: quando o horizonte é curto e o dado é pouco, o recursivo costuma vencer; quando o
erro recursivo acumula demais, o direto se paga.

## O confronto final da pasta

Os quatro candidatos, na mesma janela de avaliação: o sarrafo, o ETS, o SARIMAX e o boosting com atributos.


In [ ]:
_, passos_ingenuo = backtest(serie, lambda t, h: np.resize(t.iloc[-PERIODO:].to_numpy(), h))
_, passos_ets = backtest(serie, lambda t, h: ExponentialSmoothing(
    t, trend="add", seasonal="add", seasonal_periods=PERIODO).fit().forecast(h).to_numpy())
_, passos_sarimax = backtest(serie, lambda t, h: SARIMAX(
    t, order=(1, 1, 1), seasonal_order=(1, 1, 0, PERIODO),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False).get_forecast(h).predicted_mean.to_numpy())

comparacao = pd.DataFrame({
    "MAE médio": [passos_ingenuo.mean(), passos_ets.mean(), passos_sarimax.mean(),
                  passos_boosting.mean()],
    "MAE no passo 1": [passos_ingenuo[:, 0].mean(), passos_ets[:, 0].mean(),
                       passos_sarimax[:, 0].mean(), passos_boosting[:, 0].mean()],
    "MAE no passo 14": [passos_ingenuo[:, -1].mean(), passos_ets[:, -1].mean(),
                        passos_sarimax[:, -1].mean(), passos_boosting[:, -1].mean()],
}, index=["sazonal ingênuo", "ETS(A,A,A)", "SARIMAX(1,1,1)(1,1,0,7)",
          f"boosting ({melhor_config})"]).round(3)
display(comparacao)

comparacao[["MAE no passo 1", "MAE no passo 14"]].plot(kind="bar")
plt.title("os quatro candidatos, no primeiro e no último passo do horizonte")
plt.ylabel("MAE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


O resultado é o que a pasta vinha anunciando, e não é o que a intuição sobre aprendizado de máquina
costuma esperar: o boosting fica em **terceiro**, atrás do ETS e do SARIMAX, e à frente apenas do sarrafo.
Com o melhor conjunto de atributos, o MAE médio dele é 3,415, contra 2,947 do ETS e 3,395 do SARIMAX; o
sazonal ingênuo faz 4,040.

E a coluna do passo 14 mostra por quê: 5,362 contra 2,713 do SARIMAX. O boosting aqui é recursivo, então
cada previsão dele entra como dado para a próxima, e o erro acumula ao longo do horizonte. O SARIMAX resolve
o horizonte inteiro de uma vez e não acumula nada.

Isso não faz do boosting um método ruim: ele é o único dos quatro que aceita atributo externo, feriado,
promoção, clima, e é o que ganhou a competição M5 justamente por isso. O que este notebook mostra é que, quando
a série tem exatamente a estrutura que um método clássico supõe, o clássico ganha com muito menos trabalho.

## O que o modelo usa

O erro diz quem ganha; a importância diz por quê. A permutação embaralha uma coluna de cada vez e mede o
quanto o erro piora: coluna que não muda nada é coluna que o modelo não usa.


In [ ]:
tabela_final = tabela_de_atributos(treino, **configuracoes[melhor_config])
alvo_final = tabela_final.pop("valor")
modelo_final = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
modelo_final.fit(tabela_final, alvo_final)

importancia = permutation_importance(modelo_final, tabela_final, alvo_final,
                                     n_repeats=5, random_state=RANDOM_STATE,
                                     scoring="neg_mean_absolute_error")
ordem = np.argsort(importancia.importances_mean)[::-1]
tabela_importancia = pd.DataFrame({
    "atributo": tabela_final.columns[ordem],
    "queda no MAE": importancia.importances_mean[ordem],
}).set_index("atributo").round(3)
display(tabela_importancia)

tabela_importancia.plot(kind="barh", legend=False, color="slateblue")
plt.title("importância por permutação, dentro do treino")
plt.xlabel("queda no MAE ao embaralhar o atributo")
plt.tight_layout()
plt.show()


O primeiro lugar é do `lag_7`, e com distância: embaralhar essa coluna piora o MAE em 5,389, mais que o
triplo do segundo colocado, e é exatamente o que o gerador declarou, uma sazonalidade semanal de amplitude 6.
O modelo descobriu sozinho o componente que o ETS recebe como parâmetro e que o SARIMAX recebe como ordem
sazonal.

Depois dele vêm `media_7` e `lag_1` (1,568 e 1,390), que são dois jeitos de olhar para o passado recente, e
`lag_21` (1,132), que é o mesmo dia da semana três semanas atrás. O `dia_do_ano` aparece em quinto (1,060),
confirmando a medição anterior: entrou na tabela e está sendo usado.

Repare no fim da lista: `mes` fica em 0,091, praticamente ignorado, porque o `dia_do_ano` já carrega a
mesma informação de forma mais fina. E o `desvio_7` fica em 0,495: numa série de ruído com variância
constante, saber o quanto o passado oscilou ajuda pouco a prever o próximo valor.

Leitura prática: quando `lag_7` domina assim, o problema tem sazonalidade semanal clara e provavelmente não
precisa de rede neural nenhuma.

## Resumo

Transformar a série numa tabela é o caminho mais usado na prática, e ele traz quatro decisões que se medem.
Neste notebook: os atrasos viraram colunas com o calendário do dia; o dia do ano entrou como candidato e
passou no teste, levando o MAE médio de 3,981 para 3,415; as janelas móveis foram medidas com e sem
deslocamento, e o resultado é a lição desconfortável do notebook, porque esquecer o `shift` mudou quase nada
(3,927 contra 3,981) enquanto o atributo que contém a própria resposta derruba a validação sorteada para
perto de zero; o recursivo bateu o direto (3,981 contra 5,645) com catorze vezes menos treino; e no confronto
final os quatro candidatos ficaram lado a lado, com o ETS em 2,947, o SARIMAX em 3,395, o boosting em 3,415 e
o sazonal ingênuo em 4,040 -- e o boosting perde no passo 14 (5,362 contra 2,713) porque acumula erro.

A importância por permutação fecha o raciocínio: o `lag_7` domina, que é a sazonalidade semanal que o gerador
declarou e que os métodos clássicos recebem como parâmetro.

A conclusão prática é a mesma dos três notebooks da pasta: **o que decide o resultado não é o método mais
sofisticado, é o protocolo mais honesto**. E aqui ela vem com um corolário: quando a série tem exatamente a
estrutura que o método clássico supõe, o clássico ganha com uma fração do trabalho -- e o boosting fica com o
papel que é dele, o de aceitar informação externa que nenhum dos outros aceita.

**Próximo passo:** o notebook de redes neurais e modelo fundacional desta pasta, onde a série volta a ser
sequência em vez de tabela.

## Referências

```bibtex
@article{taieb2012review,
  title   = {A Review and Comparison of Strategies for Multi-Step Ahead Time Series Forecasting Based on the NN5 Forecasting Competition},
  author  = {Taieb, Souhaib Ben and Bontempi, Gianluca and Atiya, Amir F. and Sorjamaa, Antti},
  journal = {Expert Systems with Applications},
  volume  = {39},
  number  = {8},
  pages   = {7067--7083},
  year    = {2012},
  doi     = {10.1016/j.eswa.2012.01.039}
}

@article{makridakis2022m5,
  title   = {The M5 Accuracy Competition: Results, Findings and Conclusions},
  author  = {Makridakis, Spyros and Spiliotis, Evangelos and Assimakopoulos, Vassilios},
  journal = {International Journal of Forecasting},
  volume  = {38},
  number  = {4},
  pages   = {1346--1364},
  year    = {2022},
  doi     = {10.1016/j.ijforecast.2021.11.001}
}

@book{hyndman2021forecasting,
  title     = {Forecasting: Principles and Practice},
  author    = {Hyndman, Rob J. and Athanasopoulos, George},
  edition   = {3},
  publisher = {OTexts},
  year      = {2021},
  url       = {https://otexts.com/fpp3/}
}
```
